In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from gdt.core.data_primitives import TimeBins
from bctools.analysis import BayesianBlocksLightcurve
import os
from astropy.io import fits


#grb_name = "bn210528586.fits"
grb_name = "bn100816026.fits"
base_path = "/home/cosi/cosi/data/grb_eliza/dc3_grbs/10-100a/acs_fits/"
fits_path = base_path+grb_name

from gdt.core.background.binned import Polynomial


In [ ]:

def plot_fits_panels_with_rate(filename):

    # =========================
    # APRI FITS
    # =========================
    hdul = fits.open(filename)
    header = hdul[0].header
    
    time_start = header['DATE-OBS']
    
    data = hdul[1].data

    # =========================
    # TEMPI
    # =========================
    t_min = data["T_MIN"]

    t_max = data["T_MAX"]
    t_center = 0.5 * (t_min + t_max)
    
    
    dt = t_max - t_min


    # =========================
    # CANALI
    # =========================
    panels = {
        "SCBA_A0": data["COUNTS_SCBA_A0_G"],
        "SCBA_A1": data["COUNTS_SCBA_A1_G"],
        "SCBB_A0": data["COUNTS_SCBB_A0_G"],
        "SCBB_A1": data["COUNTS_SCBB_A1_G"],
        "SCBC_A0": data["COUNTS_SCBC_A0_G"],
        "SCBC_A1": data["COUNTS_SCBC_A1_G"],
    }

    # =========================
    # PLOT COUNTS
    # =========================
    fig1, axes1 = plt.subplots(3, 2, figsize=(12, 10), sharex=True)
    axes1 = axes1.flatten()

    for i, (name, counts) in enumerate(panels.items()):
        ax = axes1[i]
        ax.step(t_max, counts, where="pre")
        ax.set_title(f"{name} (Counts)")
        ax.set_ylabel("Counts")

    axes1[-1].set_xlabel("Time [s]")
    plt.tight_layout()
    plt.show()

    # =========================
    # PLOT RATE (counts/sec)
    # =========================
    fig2, axes2 = plt.subplots(3, 2, figsize=(12, 10), sharex=True)
    axes2 = axes2.flatten()

    for i, (name, counts) in enumerate(panels.items()):
        rate = counts / dt

        ax = axes2[i]
        ax.step(t_max, rate, where="pre")
        ax.set_title(f"{name} (Rate)")
        ax.set_ylabel("Counts/s")

    axes2[-1].set_xlabel("Time [s]")
    plt.tight_layout()
    plt.show()

    hdul.close()
    
    return t_min,t_max,panels

In [ ]:
grb_t_min,grb_t_max,grb_panels = plot_fits_panels_with_rate(fits_path)

light_curve = {"t_min":grb_t_min,"t_max":grb_t_max,"panels":grb_panels}

In [ ]:
def analyze_lc(lightcurve, p0=0.05, isRate=False,
               panels=['z0', 'z1', 'x0', 'x1', 'y0', 'y1']):
    """
    Analyze the light curve data.

    Input:
        - lightcurve: dictionary containing:
            {
                "t_min": array,
                "t_max": array,
                "panels": {
                    "SCBA_A0": array,
                    "SCBA_A1": array,
                    "SCBB_A0": array,
                    "SCBB_A1": array,
                    "SCBC_A0": array,
                    "SCBC_A1": array
                }
            }

        - p0: false alarm probability for Bayesian Blocks
        - isRate: if True, input contains rates → converted to counts
        - panels: internal panel names to analyze (x,y,z convention)

    Output:
        tuple with LC, BB results, significance, etc.
    """

    # =========================
    # TIME BINS
    # =========================
    t_min = np.asarray(lightcurve["t_min"])
    t_max = np.asarray(lightcurve["t_max"])

    # durata dei bin (non uniforme!)
    exposure = t_max - t_min

    # =========================
    # READ PANELS + MAPPING
    # =========================
    pan = lightcurve["panels"]

    # mapping SCB → xyz (definito da te)
    signal = {}

    signal["z1"] = np.asarray(pan["SCBA_A0"])
    signal["z0"] = np.asarray(pan["SCBA_A1"])

    signal["y1"] = np.asarray(pan["SCBB_A0"])
    signal["y0"] = np.asarray(pan["SCBB_A1"])

    signal["x1"] = np.asarray(pan["SCBC_A0"])
    signal["x0"] = np.asarray(pan["SCBC_A1"])

    # se sono rate → converto a counts
    if isRate:
        for k in signal:
            signal[k] = signal[k] * exposure

    # =========================
    # COSTRUZIONE LIGHT CURVES
    # =========================
    lc = {}

    for panel in panels:
        lc[panel] = TimeBins(
            signal[panel],
            t_min,
            t_max,
            exposure
        )

    # =========================
    # BEST PANEL SELECTION
    # =========================
    best_panel = None
    best_value = float('-inf')

    for panel in panels:
        value = np.max(lc[panel].counts)

        if value > best_value:
            best_value = value
            best_panel = panel

    lc_sel = lc[best_panel]

    # =========================
    # BAYESIAN BLOCKS
    # =========================
    try:
        bb_lc = BayesianBlocksLightcurve(lc_sel)
        bb_lc.compute_bayesian_blocks(p0=p0)

        signal_range = bb_lc.signal_range

        t90 = bb_lc.duration(quantile=.9)
        t90_error = bb_lc.duration_error(.9, nsamples=100)

    except Exception as e:
        print(e)
        print("WARNING")
        return lc_sel, None, -9999, -9999, -9999, -9999, -9999, -9999, -9999, None, None

    # =========================
    # SIGNAL + BACKGROUND
    # =========================
    signal_lc = lc_sel.slice(signal_range.tstart, signal_range.tstop)

    # background prima e dopo (FIX rispetto al tuo codice originale)
    bkg_lc1 = lc_sel.slice(lc_sel.centroids[0], signal_range.tstart)
    bkg_lc2 = lc_sel.slice(signal_range.tstop, lc_sel.centroids[-1])

    t_on = np.sum(signal_lc.exposure)
    t_off = np.sum(bkg_lc1.exposure) + np.sum(bkg_lc2.exposure)

    N_on = np.sum(signal_lc.rates * signal_lc.exposure)
    N_off = (
        np.sum(bkg_lc1.rates * bkg_lc1.exposure) +
        np.sum(bkg_lc2.rates * bkg_lc2.exposure)
    )

    alpha = t_on / t_off if t_off > 0 else 0

    if alpha > 0 and N_on > 0 and N_off > 0:
        S = np.sqrt(2) * (
            N_on * np.log(((1 + alpha) / alpha) * (N_on / (N_on + N_off))) +
            N_off * np.log((1 + alpha) * (N_off / (N_on + N_off)))
        )**0.5
    else:
        S = 0

    # =========================
    # PEAK SIGNIFICANCE
    # =========================
    significance = []

    for rate, exp in zip(signal_lc.rates, signal_lc.exposure):
        N_on_bin = rate * exp
        alpha_bin = exp / t_off if t_off > 0 else 0

        if alpha_bin > 0 and N_on_bin > 0 and N_off > 0:
            S_bin = np.sqrt(2) * (
                N_on_bin * np.log(((1 + alpha_bin) / alpha_bin) *
                                 (N_on_bin / (N_on_bin + N_off))) +
                N_off * np.log((1 + alpha_bin) *
                               (N_off / (N_on_bin + N_off)))
            )**0.5
        else:
            S_bin = 0

        significance.append(S_bin)

    significance = np.array(significance)
    S_peak = np.max(significance)

    # =========================
    # OUTPUT
    # =========================
    return (
        lc_sel,
        bb_lc,
        lc,
        signal_range.tstart,
        signal_range.tstop,
        t90,
        t90_error[0],
        t90_error[1],
        S,
        S_peak,
        t_min,
        t_max
    )

In [ ]:
def fit_background_gdt(lc, signal_range, buffer=0.0, order=2):
    """
    Fit del background polinomiale su una light curve GDT (TimeBins),
    escludendo la finestra del segnale.

    Parameters
    ----------
    lc : TimeBins
        Light curve del detector.
        Deve avere almeno: counts, lo_edges, hi_edges, exposure
    signal_range : tuple
        (tstart, tstop) del segnale/burst da escludere dal fit
    buffer : float, optional
        Margine extra da escludere attorno al segnale
    order : int, optional
        Ordine del polinomio

    Returns
    -------
    result : dict
        Dizionario con:
        - "model"           : oggetto Polynomial fittato
        - "mask_bkg"        : maschera booleana dei bin usati nel fit
        - "bkg_rate"        : background stimato in rate
        - "bkg_rate_err"    : errore sul background rate
        - "bkg_counts"      : background stimato in counts/bin
        - "bkg_counts_err"  : errore in counts/bin
        - "net_counts"      : counts osservati - background counts
        - "net_rate"        : rate osservato - background rate
    """

    tstart_sig = signal_range.tstart
    tstop_sig = signal_range.tstop
    excl_start = tstart_sig - buffer
    excl_stop = tstop_sig + buffer

    # bin completamente fuori dalla regione esclusa
    mask_bkg = (lc.hi_edges <= excl_start) | (lc.lo_edges >= excl_stop)

    n_bkg_bins = np.sum(mask_bkg)
    if n_bkg_bins < (order + 2):
        raise RuntimeError(
            f"Troppi pochi bin di background ({n_bkg_bins}) "
            f"per un polinomio di ordine {order}"
        )

    # costruiamo il modello come in bctools
    bkg_model = Polynomial(
        counts=lc.counts[mask_bkg][:, np.newaxis],
        tstart=lc.lo_edges[mask_bkg],
        tstop=lc.hi_edges[mask_bkg],
        exposure=lc.exposure[mask_bkg]
    )

    bkg_model.fit(order=order)

    # ATTENZIONE: interpolate() restituisce RATE, non counts
    bkg_rate, bkg_rate_err = bkg_model.interpolate(
        tstart=lc.lo_edges,
        tstop=lc.hi_edges
    )

    # da shape (N, 1) a (N,)
    bkg_rate = np.squeeze(bkg_rate)
    bkg_rate_err = np.squeeze(bkg_rate_err)

    # conversione a counts/bin
    bkg_counts = bkg_rate * lc.exposure
    bkg_counts_err = bkg_rate_err * lc.exposure

    # osservati
    obs_rate = lc.counts / lc.exposure

    # netti
    net_counts = lc.counts - bkg_counts
    net_rate = obs_rate - bkg_rate

    return {
        "model": bkg_model,
        "mask_bkg": mask_bkg,
        "bkg_rate": bkg_rate,
        "bkg_rate_err": bkg_rate_err,
        "bkg_counts": bkg_counts,
        "bkg_counts_err": bkg_counts_err,
        "net_counts": net_counts,
        "net_rate": net_rate,
    }

In [ ]:
light_curve.keys()

In [ ]:
light_curve['panels'].keys()

In [ ]:
output = analyze_lc(light_curve, p0=10e-5, isRate=False, panels=['z1', 'z0', 'x1', 'x0', 'y1', 'y0'])

In [ ]:
bb_lc = output[1]
lc_sel = output[0]
signal_range = bb_lc.signal_range


fig = plt.figure(figsize=(10,4))


plt.step(lc_sel.centroids, lc_sel.counts, where="mid")

plt.xlabel("Time [s]")
plt.ylabel("Counts / bin")
plt.title("Light curve")
plt.grid(True, alpha=0.3)

plt.show()

fig = plt.figure(figsize=(10,4))


plt.step(lc_sel.centroids, lc_sel.rates, where="mid")

plt.xlabel("Time [s]")
plt.ylabel("Counts / s")
plt.title("Light curve")
plt.grid(True, alpha=0.3)

plt.show()

fig = plt.figure(figsize=(10,4))


plt.plot(lc_sel.centroids, bb_lc.bkg_counts/lc_sel.exposure, color = 'red', ls = ':',
                label = "Fitted background")
plt.errorbar(lc_sel.centroids, lc_sel.rates, xerr = [lc_sel.centroids-lc_sel.lo_edges, 
 lc_sel.hi_edges-lc_sel.centroids],
                    yerr = lc_sel.rate_uncertainty, 
                    ls = 'none', color = '.7',
                    label = 'Raw data')

lc_bayes = bb_lc.bb_lightcurve

plt.plot(np.append(lc_bayes.lo_edges, lc_bayes.hi_edges[-1]),
                np.append(lc_bayes.rates, lc_bayes.rates[-1]),
                drawstyle = 'steps-post',
                label = 'Bayesian blocks')
        
# Vertical lines showing the start and stop of the identified signal
plt.axvline(bb_lc.signal_range.tstart, ls = "--", color = 'olive', label = "Signal start/stop")
plt.axvline(bb_lc.signal_range.tstop, ls = "--", color = 'olive')

plt.legend()



In [ ]:
# =========================
# ZOOM ATTORNO AL SEGNALE
# =========================
tstart = signal_range.tstart
tstop  = signal_range.tstop

# margine automatico (20% della durata, minimo 1s)
margin = max(1.0, 1.0 * (tstop - tstart))

xmin = tstart - margin
xmax = tstop + margin

# =========================
# PLOT ZOOM (RATE)
# =========================
fig = plt.figure(figsize=(10,4))

# dati
plt.errorbar(
    lc_sel.centroids,
    lc_sel.rates,
    xerr=[lc_sel.centroids - lc_sel.lo_edges,
          lc_sel.hi_edges - lc_sel.centroids],
    yerr=lc_sel.rate_uncertainty,
    ls='none', color='.7',
    label='Data'
)

# background
plt.plot(
    lc_sel.centroids,
    bb_lc.bkg_counts / lc_sel.exposure,
    color='red', ls=':',
    label='Background'
)

# Bayesian Blocks
lc_bayes = bb_lc.bb_lightcurve

plt.plot(
    np.append(lc_bayes.lo_edges, lc_bayes.hi_edges[-1]),
    np.append(lc_bayes.rates, lc_bayes.rates[-1]),
    drawstyle='steps-post',
    label='Bayesian blocks'
)

# linee segnale
plt.axvline(tstart, ls="--", color="olive", label="Signal start/stop")
plt.axvline(tstop,  ls="--", color="olive")

# 🔥 ZOOM
plt.xlim(xmin, xmax)

plt.xlabel("Time [s]")
plt.ylabel("Counts / s")
plt.title("Zoom around signal")
plt.legend()
plt.grid(True, alpha=0.3)

plt.show()

In [ ]:
lc_array = output[2]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

panels = ['z1','z0','x1','x0','y1','y0']

results = []
signal_counts_arr = []
background_counts_arr = []
net_counts_arr = []

tstart = signal_range.tstart
tstop = signal_range.tstop
duration = tstop-tstart
for p in panels:
    
    print("Panel:", p)
   
    lc = lc_array[p]
    res = fit_background_gdt(lc, signal_range, buffer=1.0, order=2)

    results.append(res)
    
    # maschera della finestra del segnale
    mask_sig = (lc.lo_edges >= tstart) & (lc.hi_edges <= tstop)

    # somme nella finestra [tstart, tstop]
    signal_counts = np.sum(lc.counts[mask_sig])
    background_counts = np.sum(res["bkg_counts"][mask_sig])
    net_counts = signal_counts - background_counts

    signal_counts_arr.append(signal_counts)
    background_counts_arr.append(round(background_counts,2))
    net_counts_arr.append(net_counts)
    
    # plot
    t = 0.5 * (lc.lo_edges + lc.hi_edges)
    obs_rate = lc.counts / lc.exposure

    plt.figure(figsize=(10, 5))

    # counts osservati
    plt.step(t, lc.rates, where="mid", label="Observed counts")

    # background in counts
    plt.plot(t, res["bkg_rate"], label="Background (counts)")

    plt.axvspan(signal_range.tstart, signal_range.tstop, alpha=0.2, label="Signal window")

    plt.xlabel("Time")
    plt.ylabel("Counts / bin")
    

    plt.title(
        f"{p} | signal={signal_counts:.1f}, "
        f"bkg={background_counts:.1f}, net={net_counts:.1f}"
    )

    plt.legend()
    plt.show()
# conversione finale in array numpy
signal_counts_arr = np.array(signal_counts_arr)
background_counts_arr = np.array(background_counts_arr)
net_counts_arr = np.array(net_counts_arr)

print("Signal counts per panel:", signal_counts_arr)
print("Background counts per panel:", background_counts_arr)
print("Net counts per panel:", net_counts_arr)